# SORA.Earth - SHAP Model Interpretability

In [ ]:
import pandas as pd
import numpy as np
import pickle
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

## 1. Загрузка

In [ ]:
df=pd.read_csv('../data/projects.csv')
FEAT=['budget','co2_reduction','social_impact','duration_months']
X=df[FEAT];y=df['success']
with open('../models/model.pkl','rb') as f: rf=pickle.load(f)
with open('../models/xgb_model.pkl','rb') as f: xgb=pickle.load(f)
with open('../models/scaler.pkl','rb') as f: scaler=pickle.load(f)
X_sc=pd.DataFrame(scaler.transform(X),columns=FEAT)
print('OK',X_sc.shape)

## 1. Загрузка

In [ ]:
df=pd.read_csv('../data/projects.csv')
FEAT=['budget','co2_reduction','social_impact','duration_months']
X=df[FEAT];y=df['success']
with open('../models/model.pkl','rb') as f: rf=pickle.load(f)
with open('../models/xgb_model.pkl','rb') as f: xgb=pickle.load(f)
with open('../models/scaler.pkl','rb') as f: scaler=pickle.load(f)
X_sc=pd.DataFrame(scaler.transform(X),columns=FEAT)
print('OK',X_sc.shape)

## 2. SHAP XGBoost (основная модель)

In [ ]:
exp=shap.TreeExplainer(xgb)
sv=exp.shap_values(X_sc)
print('Shape:',sv.shape)

## 3. Summary Plot

In [ ]:
shap.summary_plot(sv,X_sc,show=False)
plt.savefig('../plots/shap_summary.png',dpi=150,bbox_inches='tight')
plt.show()

## 4. Bar Plot

In [ ]:
shap.summary_plot(sv,X_sc,plot_type='bar',show=False)
plt.savefig('../plots/shap_bar.png',dpi=150,bbox_inches='tight')
plt.show()

## 5. Dependence Plots

In [ ]:
fig,axes=plt.subplots(2,2,figsize=(14,10))
for i,f in enumerate(FEAT):
    shap.dependence_plot(f,sv,X_sc,ax=axes.flat[i],show=False)
plt.tight_layout()
plt.savefig('../plots/shap_dep.png',dpi=150,bbox_inches='tight')
plt.show()

## 6. Waterfall (пример)

In [ ]:
bv=float(exp.expected_value) if np.isscalar(exp.expected_value) else float(exp.expected_value[0])
e=shap.Explanation(values=sv[0],base_values=bv,data=X_sc.iloc[0].values,feature_names=FEAT)
shap.plots.waterfall(e,show=False)
plt.savefig('../plots/shap_waterfall.png',dpi=150,bbox_inches='tight')
plt.show()

## 7. Force Plot (top-5)

In [ ]:
for i in range(5):
    e=shap.Explanation(values=sv[i],base_values=bv,data=X_sc.iloc[i].values,feature_names=FEAT)
    print(f'{df.iloc[i][chr(110)+chr(97)+chr(109)+chr(101)]}: pred_contrib={sv[i].round(3)}')

## 8. Средняя важность (SHAP)

In [ ]:
imp=np.abs(sv).mean(axis=0)
for f,v in sorted(zip(FEAT,imp),key=lambda x:-x[1]):
    print(f'{f:20s} mean|SHAP|={v:.4f}')